# 00 論文用コア確認ノートブック

最終更新: 2026-07-21 09:58:48 JST

論文で説明する実験の最小構成、使用ソースコード、確認観点をまとめるNotebookです。

このNotebookは研究全体の一部です。単独でも動きますが、論文用には以下の順番で確認します。

```text
00 paper_core_experiment.ipynb       : 論文用の最小実験整理
01 student_ai_colab.ipynb            : 生徒AIの設計確認
02 student_ai_colab.ipynb            : 理解度と正答率の学習曲線確認
03 personality_experiment.ipynb      : 個人特徴と伝達AI分類
04 teaching_strategy_experiment.ipynb: 複数生徒クラスと授業方略
```


<!-- notebook-role-guide -->
## このNotebookで確認すること

- 研究のコア実験だけを確認する
- 使用するソースコードと実験条件を対応づける
- 論文に載せる比較表や確認ポイントを作る

## 実行順

- Colabでは 0 を先に実行する
- 実験の概要確認は 1-4 を実行する
- 授業構成比較は 6-9 を実行する
- LLMの確認は 5 を必要な時だけ実行する

## 主な出力

- 論文用の実験条件
- クラス条件ごとの授業構成比較
- 論文に使う確認ポイント

## 注意

- ColabでGit更新セルを実行しても、すでに開いているNotebook画面のセル内容は自動では書き換わりません。
- Notebook自体を更新した場合は、GitHub上の最新版Notebookを開き直してください。
- LLMを使うセルはロードに時間がかかります。まずはmockまたは軽量モデルで確認してください。


## 0. Colab setup

Run this cell first in Colab. It clones or updates the GitHub repository, moves to the repository root, and makes `src` importable.

In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai
!git pull
!git log -1 --oneline
%pip install -q -r requirements.txt

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("project_root:", project_root)
print("src import path ready:", str(project_root) in sys.path)

## 1. 使用するソースコード

論文で説明する処理と、実装ファイルの対応を確認します。

In [ ]:
from pprint import pprint

SOURCE_CODE_MAP = [
    {"file": "src/class_manager.py", "role": "クラス定義と所属生徒を読み込む"},
    {"file": "src/observer/observation_filter.py", "role": "授業中に観察可能なイベントを作る"},
    {"file": "src/observer/trait_classifier.py", "role": "伝達AIとして個人反応とクラス全体を要約する"},
    {"file": "src/teacher/belief_manager.py", "role": "教師側の推定状態を更新する"},
    {"file": "src/teacher/lesson_planner.py", "role": "クラス状態から授業構成を作る"},
    {"file": "data/classes/*.json", "role": "クラス条件を定義する"},
    {"file": "data/students/*.json", "role": "生徒AIの内部状態を定義する"},
]

try:
    import pandas as pd
    display(pd.DataFrame(SOURCE_CODE_MAP))
except ImportError:
    pprint(SOURCE_CODE_MAP)

## 2. 実験設定

論文用の主実験では、低理解クラス、混合クラス、高理解クラスの3条件で授業構成が変わるかを比較します。

In [ ]:
import json
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.class_manager import ClassManager
from src.observer.observation_filter import build_observable_event
from src.teacher.lesson_planner import RuleBasedLessonPlanner

OUTPUT_DIR = Path("data/assessments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_MINUTES = 30
CLASS_IDS_FOR_DATASET_CHECK = ["class_3_basic", "class_10_mixed", "class_20_mixed"]

CURRICULUM_FOR_PAPER = {
    "lesson_goals": [
        {
            "goal_id": "transpose_terms",
            "target_skill": "can_transpose_terms",
            "goal_text": "移項すると符号が変わることを理解する",
            "success_criteria": ["x + a = b 型で a を反対側へ移せる"],
        },
        {
            "goal_id": "divide_by_coefficient",
            "target_skill": "can_divide_by_coefficient",
            "goal_text": "係数で両辺を割って x を求める",
            "success_criteria": ["ax = b 型で両辺を a で割れる"],
        },
    ],
    "next_problem_bank": {
        "can_transpose_terms": [
            {"problem": "x + 3 = 8", "answer": "x = 5"},
            {"problem": "x - 4 = 6", "answer": "x = 10"},
        ],
        "can_divide_by_coefficient": [
            {"problem": "3x = 15", "answer": "x = 5"},
            {"problem": "5x = 20", "answer": "x = 4"},
        ],
    },
}

print("total_minutes:", TOTAL_MINUTES)
print("dataset_classes:", CLASS_IDS_FOR_DATASET_CHECK)

## 3. データセットとして用意したクラスを確認

`data/classes/*.json` と `data/students/*.json` から、実験に使えるクラス定義を読み込みます。

In [ ]:
class_manager = ClassManager()

class_dataset_rows = []
for class_id in CLASS_IDS_FOR_DATASET_CHECK:
    summary = class_manager.summarize_class(class_id)
    class_dataset_rows.append(
        {
            "class_id": class_id,
            "student_count": summary["student_count"],
            "average_score": summary["average_score"],
            "score_std": summary["score_std"],
            "low_score_count": len(summary["low_score_students"]),
            "high_score_count": len(summary["high_score_students"]),
            "misconception_count": summary["misconception_count"],
            "features": summary["class_features"].get("understanding_profile"),
        }
    )

class_dataset_table = pd.DataFrame(class_dataset_rows)
display(class_dataset_table)

## 4. 情報分離の確認

教師AIに渡す前の観察可能イベントには、真の `knowledge_state` や `big_five` などの内部状態を入れないことを確認します。

In [ ]:
observable_event = build_observable_event(
    lesson_id="PAPER_L001",
    teacher_id="T001",
    student_id="S001",
    teacher_prompt="x + 3 = 8 を解いてください。",
    utterance="x = 5 です。3を反対側に移すと -3 になります。",
    answer="x = 5",
    is_correct=True,
    response_time_sec=3.2,
).to_dict()

forbidden_internal_fields = {
    "knowledge_state",
    "understanding",
    "big_five",
    "personality",
    "motivation",
    "misconceptions",
    "error_tendency",
    "learning_speed",
}
observable_keys = set(observable_event)
leaked_fields = sorted(observable_keys & forbidden_internal_fields)

print("observable_keys:", sorted(observable_keys))
print("leaked_internal_fields:", leaked_fields)
assert leaked_fields == []

## 5. LLM communication AI bridge check

This cell checks whether the communication AI can sit between observable classroom events and the teacher AI. When `USE_LLM_COMMUNICATION_AI = True`, the communication AI uses a local LLM through Transformers. If the LLM output is not valid JSON, the implementation falls back to the rule-based communication AI.

In [ ]:
import importlib

from src.config import GenerationConfig, ModelLoadConfig
from src.model_loader import LocalLLM
from src.observer import CommunicationAI, LLMCommunicationAI, events_to_communication_rows
from src.teacher.belief_manager import TeacherBeliefManager

USE_LLM_COMMUNICATION_AI = True
COMMUNICATION_MODEL_ID = "Qwen/Qwen3-4B"
LLM_LOAD_IN_4BIT = True

def ensure_bitsandbytes():
    try:
        from transformers.utils import is_bitsandbytes_available
        if is_bitsandbytes_available():
            return True
    except Exception:
        pass

    print("bitsandbytes is not available to Transformers. Installing bitsandbytes>=0.46.1 ...")
    %pip install -q -U "bitsandbytes>=0.46.1"
    importlib.invalidate_caches()

    try:
        from transformers.utils import is_bitsandbytes_available
        available = is_bitsandbytes_available()
    except Exception as exc:
        print("Could not check bitsandbytes availability:", repr(exc))
        available = False

    if not available:
        print("bitsandbytes is installed but Transformers still cannot use it.")
        print("In Colab, restart the runtime and run from '0. Colab setup' again.")
    return available

if USE_LLM_COMMUNICATION_AI and LLM_LOAD_IN_4BIT:
    LLM_LOAD_IN_4BIT = ensure_bitsandbytes()
    print("LLM_LOAD_IN_4BIT:", LLM_LOAD_IN_4BIT)

llm_bridge_events = [
    build_observable_event(
        lesson_id="PAPER_LLM_BRIDGE",
        teacher_id="T001",
        student_id="S001",
        teacher_prompt="Solve x + 3 = 8.",
        utterance="I'm not sure. If I move +3, does the sign really become -3? Is x = 5 correct?",
        answer="x = 5",
        is_correct=True,
        response_time_sec=8.4,
    ).to_dict(),
    build_observable_event(
        lesson_id="PAPER_LLM_BRIDGE",
        teacher_id="T001",
        student_id="S002",
        teacher_prompt="Solve x + 3 = 8.",
        utterance="First I move +3 to the other side, then it becomes -3, so x = 5.",
        answer="x = 5",
        is_correct=True,
        response_time_sec=4.1,
    ).to_dict(),
    build_observable_event(
        lesson_id="PAPER_LLM_BRIDGE",
        teacher_id="T001",
        student_id="S003",
        teacher_prompt="Solve x + 3 = 8.",
        utterance="x = 5",
        answer="x = 5",
        is_correct=True,
        response_time_sec=2.0,
    ).to_dict(),
]

communication_rows = events_to_communication_rows(llm_bridge_events)

if USE_LLM_COMMUNICATION_AI:
    reuse_loaded_model = (
        "communication_text_generator" in globals()
        and getattr(communication_text_generator, "model_id", None) == COMMUNICATION_MODEL_ID
        and getattr(communication_text_generator, "load_in_4bit", None) == LLM_LOAD_IN_4BIT
    )
    if reuse_loaded_model:
        print("reuse loaded communication model:", COMMUNICATION_MODEL_ID)
    else:
        print("load communication model:", COMMUNICATION_MODEL_ID)
        communication_text_generator = LocalLLM(
            model_id=COMMUNICATION_MODEL_ID,
            load_in_4bit=LLM_LOAD_IN_4BIT,
            generation_config=GenerationConfig(
                max_new_tokens=256,
                temperature=0.1,
                top_p=0.9,
                do_sample=False,
                repetition_penalty=1.05,
            ),
            model_load_config=ModelLoadConfig(load_in_4bit=LLM_LOAD_IN_4BIT),
        )
    communication_ai_for_paper = LLMCommunicationAI(communication_text_generator)
else:
    communication_ai_for_paper = CommunicationAI()

try:
    llm_classroom_observation = communication_ai_for_paper.summarize_classroom(communication_rows).to_dict()
    llm_bridge_mode = "LLMCommunicationAI" if USE_LLM_COMMUNICATION_AI else "CommunicationAI"
except Exception as exc:
    print("LLM bridge failed:", repr(exc))
    print("Falling back to rule-based CommunicationAI so the pipeline can continue.")
    communication_ai_for_paper = CommunicationAI()
    llm_classroom_observation = communication_ai_for_paper.summarize_classroom(communication_rows).to_dict()
    llm_bridge_mode = "CommunicationAI fallback"

llm_communication_results = llm_classroom_observation["individual_results"]

belief_manager = TeacherBeliefManager(base_dir=OUTPUT_DIR / "paper_teacher_beliefs_runtime")
llm_bridge_teacher_beliefs = belief_manager.update_many(
    teacher_id="T_PAPER",
    observations=llm_bridge_events,
    communication_results=llm_communication_results,
    save=False,
)

llm_bridge_lesson_plan = RuleBasedLessonPlanner().plan_lesson(
    teacher_beliefs=llm_bridge_teacher_beliefs,
    curriculum=CURRICULUM_FOR_PAPER,
    total_minutes=TOTAL_MINUTES,
)

print("communication_ai:", llm_bridge_mode)
print("student_count:", llm_classroom_observation["student_count"])
print("classroom_summary:", llm_classroom_observation["classroom_summary"])
print("teacher_belief_students:", sorted(llm_bridge_teacher_beliefs))
print("lesson_goal:", llm_bridge_lesson_plan["lesson_goal"]["goal_text"])

display(pd.DataFrame(llm_communication_results)[[
    "student_id",
    "profile_prediction",
    "confidence",
    "teacher_summary",
]])

assert llm_classroom_observation["student_count"] == 3
assert sorted(llm_bridge_teacher_beliefs) == ["S001", "S002", "S003"]

## 6. 論文用のクラス条件を作る

ここでは比較しやすいように、教師側の推定状態 `teacher_belief` を3条件に固定します。これは、観察結果が蓄積された後に教師AIが持つ推定状態を想定した入力です。

In [ ]:
def make_belief(score, *, self_efficacy="medium", question_tendency="medium", motivation="medium", neuroticism="medium", misconceptions=None):
    return {
        "estimated_knowledge": {"linear_equation": {"score": score, "confidence": 0.7}},
        "estimated_traits": {
            "self_efficacy": {"level": self_efficacy, "confidence": 0.7},
            "question_tendency": {"level": question_tendency, "confidence": 0.7},
            "motivation": {"level": motivation, "confidence": 0.7},
            "conscientiousness": {"level": "medium", "confidence": 0.7},
            "neuroticism": {"level": neuroticism, "confidence": 0.7},
        },
        "estimated_misconceptions": misconceptions or [],
    }

coefficient_misconception = [
    {"name": "3x = 15 で 3 を引けばよいと考える", "confidence": 0.8, "evidence_count": 2}
]

PAPER_CLASS_CONDITIONS = {
    "low_understanding_class": {
        "S001": make_belief(28, self_efficacy="low", question_tendency="low", motivation="low", neuroticism="high"),
        "S002": make_belief(35, self_efficacy="low", question_tendency="medium", neuroticism="high"),
        "S003": make_belief(42, question_tendency="low"),
    },
    "mixed_class": {
        "S001": make_belief(35, self_efficacy="low", neuroticism="high"),
        "S002": make_belief(62),
        "S003": make_belief(88, self_efficacy="high", question_tendency="low"),
    },
    "coefficient_misconception_class": {
        "S001": make_belief(62, misconceptions=coefficient_misconception),
        "S002": make_belief(68, question_tendency="low"),
        "S003": make_belief(72, self_efficacy="high"),
    },
    "high_understanding_class": {
        "S001": make_belief(76, self_efficacy="high"),
        "S002": make_belief(82),
        "S003": make_belief(90, self_efficacy="high", motivation="high"),
    },
}

print("conditions:", list(PAPER_CLASS_CONDITIONS))

## 7. 授業構成を生成して比較する

`RuleBasedLessonPlanner` を使い、クラス条件ごとの授業目標と時間配分を比較します。

In [ ]:
PHASE_NAMES = ["導入", "全体説明", "例題", "個別演習", "確認"]

def normalize_lesson_structure(plan):
    rows = []
    for index, phase in enumerate(plan["lesson_structure"]):
        rows.append(
            {
                "phase_index": index + 1,
                "phase": PHASE_NAMES[index] if index < len(PHASE_NAMES) else str(phase.get("phase")),
                "minutes": phase.get("minutes"),
                "problem": phase.get("problem"),
                "expected_answer": phase.get("expected_answer"),
            }
        )
    return rows

def summarize_risks(profile):
    risks = []
    if profile.get("low_score_students"):
        risks.append("低理解度の生徒がいる")
    if profile.get("trait_counts", {}).get("question_tendency", {}).get("low", 0) >= 2:
        risks.append("質問しにくい生徒が複数いる")
    if profile.get("trait_counts", {}).get("self_efficacy", {}).get("low", 0) >= 2:
        risks.append("自己効力感が低い生徒が複数いる")
    if profile.get("common_misconceptions"):
        risks.append("共通誤概念がある")
    return " / ".join(risks) if risks else "大きな共通リスクなし"

planner = RuleBasedLessonPlanner()
lesson_plans = {}
comparison_rows = []
phase_rows = []

for condition_name, teacher_beliefs in PAPER_CLASS_CONDITIONS.items():
    plan = planner.plan_lesson(
        teacher_beliefs=teacher_beliefs,
        curriculum=CURRICULUM_FOR_PAPER,
        total_minutes=TOTAL_MINUTES,
    )
    profile = plan["class_profile"]
    structure = normalize_lesson_structure(plan)
    lesson_plans[condition_name] = {
        "lesson_goal": plan["lesson_goal"],
        "class_profile": {
            "student_count": profile["student_count"],
            "average_estimated_score": profile["average_estimated_score"],
            "score_std": profile["score_std"],
            "low_score_students": profile["low_score_students"],
            "high_score_students": profile["high_score_students"],
            "trait_counts": profile["trait_counts"],
            "risk_summary": summarize_risks(profile),
        },
        "lesson_structure": structure,
        "individual_support_count": len(plan["individual_support_policy"]),
    }
    phase_minutes = {row["phase"]: row["minutes"] for row in structure}
    comparison_rows.append(
        {
            "condition": condition_name,
            "student_count": profile["student_count"],
            "average_estimated_score": profile["average_estimated_score"],
            "score_std": profile["score_std"],
            "low_score_count": len(profile["low_score_students"]),
            "high_score_count": len(profile["high_score_students"]),
            "lesson_goal": plan["lesson_goal"]["goal_text"],
            "target_skill": plan["lesson_goal"]["target_skill"],
            "intro_min": phase_minutes.get("導入"),
            "explanation_min": phase_minutes.get("全体説明"),
            "worked_example_min": phase_minutes.get("例題"),
            "practice_min": phase_minutes.get("個別演習"),
            "check_min": phase_minutes.get("確認"),
            "individual_support_count": len(plan["individual_support_policy"]),
            "risk_summary": summarize_risks(profile),
        }
    )
    for row in structure:
        phase_rows.append({"condition": condition_name, **row})

lesson_comparison_table = pd.DataFrame(comparison_rows)
lesson_phase_table = pd.DataFrame(phase_rows)

display(lesson_comparison_table)
display(lesson_phase_table)

## 8. 論文用に確認するポイント

低理解クラスでは基礎説明を厚くし、混合クラスでは個別演習が長くなり、誤概念クラスでは係数処理が目標になる、という差が出ているかを確認します。

In [ ]:
checks = {
    "low_class_uses_transposition_goal": lesson_plans["low_understanding_class"]["lesson_goal"]["target_skill"] == "can_transpose_terms",
    "misconception_class_uses_coefficient_goal": lesson_plans["coefficient_misconception_class"]["lesson_goal"]["target_skill"] == "can_divide_by_coefficient",
    "low_class_explanation_is_longer_than_high_class": (
        lesson_comparison_table.set_index("condition").loc["low_understanding_class", "explanation_min"]
        > lesson_comparison_table.set_index("condition").loc["high_understanding_class", "explanation_min"]
    ),
}

pprint(checks)
assert all(checks.values())

## 9. 結果を保存する

論文用の表とJSONを保存します。

In [ ]:
comparison_csv = OUTPUT_DIR / "paper_lesson_plan_comparison.csv"
phase_csv = OUTPUT_DIR / "paper_lesson_phase_table.csv"
plans_json = OUTPUT_DIR / "paper_lesson_plans.json"
source_map_json = OUTPUT_DIR / "paper_source_code_map.json"

lesson_comparison_table.to_csv(comparison_csv, index=False)
lesson_phase_table.to_csv(phase_csv, index=False)
plans_json.write_text(json.dumps(lesson_plans, ensure_ascii=False, indent=2), encoding="utf-8")
source_map_json.write_text(json.dumps(SOURCE_CODE_MAP, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved:", comparison_csv)
print("saved:", phase_csv)
print("saved:", plans_json)
print("saved:", source_map_json)